# EDEN-03 — Spatial processing | v1.0
**Phase 1: Data engineering · Notebook 03 of 15 · CPU only**

EDEN-02 → **EDEN-03** → EDEN-04. This notebook checks the 36 saved RGB-D pairs, makes a simple green-pixel canopy mask, summarizes the original depth codes inside that mask, and records image-quality flags. The mask is a first-pass visual aid, not a trained plant detector. Calibration and depth units are not verified, so outputs stay in image pixels, proportions, and native depth codes—no centimetres, plant height, or 3D geometry.

**Input:** attach `EDEN-02-RGBD-Acquisition-Results-v1.1.zip` as a Kaggle input. **Output:** `EDEN-03-Spatial-Processing-Results-v1.0.zip` with per-image features, masks, QC reports, preview montage, provenance, and manifest. Keep EDEN-01 attached separately for later climate work.

Run cells in order on **CPU, Internet Off**. Upload the named ZIP here after the run; then we can review its real outputs before moving to EDEN-04.

## Cell 1 — Set folders and image-space rules

This cell imports the tools, finds Kaggle input/output folders, and sets the green-pixel thresholds. The thresholds are simple starting points that you can adjust after looking at the montage. `image-space` means measurements in pixels or proportions, without claiming real-world size.

In [ ]:
# Import tools for paths, tables, images, hashing, and plots.
from pathlib import Path
import hashlib, json, zipfile, shutil, warnings
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

# Kaggle puts attached datasets under /kaggle/input and writable files under /kaggle/working.
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
RESULT_NAME = 'EDEN-03-Spatial-Processing-Results-v1.0.zip'
OUT_ROOT = WORK_ROOT / 'eden03'
SOURCE_ROOT = WORK_ROOT / 'eden02_input'

# Keep these first-pass thresholds visible so they can be reviewed and changed.
GREEN_EXG_MIN = 15.0       # Excess Green score above this is a possible plant pixel.
GREEN_G_DOMINANCE = 0.80   # Green should be at least this fraction of red + blue.
DARK_MEAN_THRESHOLD = 5.0  # Very dark RGB frames are flagged for human review.
PREVIEW_WIDTH = 960        # Smaller previews reduce notebook memory use.

# Recreate this notebook's output folders for a clean, repeatable run.
if OUT_ROOT.exists():
    shutil.rmtree(OUT_ROOT)
for folder in ['reports', 'vision/masks']:
    (OUT_ROOT / folder).mkdir(parents=True, exist_ok=True)
print('EDEN-03 folders ready:', OUT_ROOT)

## Cell 2 — Find and verify the EDEN-02 result

The manifest is the result bundle's file receipt: it lists each expected file and its SHA-256 fingerprint. This cell finds exactly one EDEN-02 v1.1 result, checks every listed file, and unpacks it. If Kaggle attached more than one matching bundle, the notebook stops so the input can be made unambiguous.

In [ ]:
# Find either the attached result ZIP or an already extracted result folder.
zip_candidates = list(INPUT_ROOT.rglob('EDEN-02-RGBD-Acquisition-Results-v1.1.zip'))
manifest_candidates = list(INPUT_ROOT.rglob('manifest.json'))
source_manifest_path = None
source_zip = None

# Prefer a specifically named result ZIP when it is attached.
if len(zip_candidates) == 1:
    source_zip = zip_candidates[0]
elif len(zip_candidates) > 1:
    raise RuntimeError(f'Found multiple EDEN-02 ZIP files: {zip_candidates}')
else:
    # Otherwise, locate an extracted EDEN-02 manifest by its producer fields.
    for candidate in manifest_candidates:
        try:
            data = json.loads(candidate.read_text(encoding='utf-8'))
            if data.get('producer') == 'EDEN-02' and data.get('code_version') == '1.1':
                if source_manifest_path is not None:
                    raise RuntimeError('Found multiple extracted EDEN-02 v1.1 outputs.')
                source_manifest_path = candidate
        except (OSError, json.JSONDecodeError):
            continue
    if source_manifest_path is None:
        raise FileNotFoundError('Attach EDEN-02-RGBD-Acquisition-Results-v1.1.zip to Kaggle.')

# Read the manifest and verify each listed file before using it.
if source_zip:
    with zipfile.ZipFile(source_zip) as archive:
        manifest_name = next((n for n in archive.namelist() if n.endswith('/manifest.json')), None)
        if manifest_name is None:
            raise FileNotFoundError('The attached ZIP has no manifest.json.')
        source_manifest = json.loads(archive.read(manifest_name))
        prefix = manifest_name.rsplit('/', 1)[0]
        assert source_manifest.get('producer') == 'EDEN-02'
        assert source_manifest.get('code_version') == '1.1'
        assert source_manifest.get('status') == 'PASS_RGBD_PILOT'
        verified_count = 0
        for item in source_manifest['files']:
            member = f"{prefix}/{item['path']}"
            payload = archive.read(member)
            assert len(payload) == item['size'], f"Size mismatch: {member}"
            assert hashlib.sha256(payload).hexdigest() == item['sha256'], f"Hash mismatch: {member}"
            verified_count += 1
        archive.extractall(SOURCE_ROOT)
        SOURCE_ROOT = SOURCE_ROOT / prefix
else:
    # The extracted-folder option checks the same receipt without unpacking.
    source_manifest = json.loads(source_manifest_path.read_text(encoding='utf-8'))
    assert source_manifest.get('producer') == 'EDEN-02'
    assert source_manifest.get('code_version') == '1.1'
    assert source_manifest.get('status') == 'PASS_RGBD_PILOT'
    SOURCE_ROOT = source_manifest_path.parent
    verified_count = 0
    for item in source_manifest['files']:
        file_path = SOURCE_ROOT / item['path']
        assert file_path.stat().st_size == item['size'], f"Size mismatch: {file_path}"
        assert hashlib.sha256(file_path.read_bytes()).hexdigest() == item['sha256'], f"Hash mismatch: {file_path}"
        verified_count += 1

print(f"Verified {verified_count} EDEN-02 files; source status: {source_manifest['status']}")

## Cell 3 — Load the paired-image index and check paths

The image index links each RGB image to its matching depth image and keeps its compartment, timestamp, and split. A `split` is the frozen train/validation/test group from EDEN-01. This cell checks the expected 36 unique pairs and confirms both image files exist before processing.

In [ ]:
# Load the EDEN-02 image index and keep compartment IDs as text.
images = pd.read_csv(SOURCE_ROOT / 'vision/images.csv', dtype={'compartment_id': str})
assert images['image_id'].is_unique, 'Image IDs are not unique.'
assert len(images) == 36, f'Expected the saved 36-pair pilot, found {len(images)}.'
assert set(images['split'].dropna()) <= {'train', 'validation', 'test'}

# Check the indexed RGB and depth files exist before decoding them.
for _, row in images.iterrows():
    assert (SOURCE_ROOT / row['rgb_path']).is_file(), f"Missing RGB: {row['rgb_path']}"
    assert (SOURCE_ROOT / row['depth_path']).is_file(), f"Missing depth: {row['depth_path']}"

# Read calibration metadata as evidence only; candidates do not prove metric calibration.
calibration_index = json.loads((SOURCE_ROOT / 'vision/calibration/index.json').read_text(encoding='utf-8'))
numeric_calibration_verified = all(x.get('numeric_calibration_verified') is True for x in calibration_index)
depth_unit = str(images['depth_unit'].dropna().iloc[0]) if images['depth_unit'].notna().any() else 'unknown'
print(f'Loaded {len(images)} pairs; numeric calibration verified: {numeric_calibration_verified}; depth unit: {depth_unit}')

## Cell 4 — Measure image quality and keep the known exceptions visible

This cell checks RGB brightness and reports depth-code patterns. The two nearly black frames found during EDEN-02 review are identified by compartment and local time so they cannot disappear inside an average. Zero and 65535 are counted as codes only—their meaning has not been confirmed, so neither is silently removed or called invalid.

In [ ]:
# These two nearly black frames were confirmed from the saved EDEN-02 pixels.
known_dark_frames = {('3.01', '2024-09-03 04:06:02'), ('3.03', '2024-09-03 04:04:01')}
qc_rows = []

# Inspect each RGB/depth pair while preserving native depth values.
for _, row in images.iterrows():
    rgb = np.asarray(Image.open(SOURCE_ROOT / row['rgb_path']).convert('RGB'))
    depth = np.asarray(Image.open(SOURCE_ROOT / row['depth_path']))
    assert depth.ndim == 2 and np.issubdtype(depth.dtype, np.integer), f"Unexpected depth array: {row['image_id']}"
    assert rgb.shape[:2] == depth.shape, f"RGB/depth dimensions differ: {row['image_id']}"
    rgb_mean = float(rgb.mean())
    rgb_max = int(rgb.max())
    zero_fraction = float(np.mean(depth == 0))
    code_65535_fraction = float(np.mean(depth == 65535)) if depth.dtype == np.uint16 else np.nan
    key = (str(row['compartment_id']), str(row['local_time']))
    flags = []
    if rgb_mean < DARK_MEAN_THRESHOLD:
        flags.append('very_dark_rgb')
    if key in known_dark_frames:
        flags.append('previously_reviewed_dark_frame')
    qc_rows.append({
        'image_id': row['image_id'], 'compartment_id': str(row['compartment_id']),
        'local_time': row['local_time'], 'split': row['split'], 'rgb_mean': rgb_mean,
        'rgb_max': rgb_max, 'depth_dtype': str(depth.dtype), 'depth_zero_fraction': zero_fraction,
        'depth_65535_fraction': code_65535_fraction, 'qc_flags': '|'.join(flags) or 'none'
    })

# Keep the check results as a table so every flagged image can be reviewed.
qc = pd.DataFrame(qc_rows)
qc.to_csv(OUT_ROOT / 'reports/image_quality.csv', index=False)
print(qc.loc[qc.qc_flags != 'none', ['image_id','compartment_id','local_time','rgb_mean','rgb_max','qc_flags']].to_string(index=False))

## Cell 5 — Make a first-pass green-pixel mask

The mask highlights pixels that look green using two simple color checks: **HSV hue** (a color wheel) and **Excess Green** (a score comparing green against red and blue). This is a hand-set heuristic, not a learned or validated plant segmentation model. It writes a reduced-size black-and-white preview mask; it does not change source images or depth.

In [ ]:
# Convert RGB values to hue/saturation/value without changing the source image.
def rgb_to_hsv(rgb):
    # Scale colors to 0..1, then calculate the brightest, darkest, and channel spread.
    x = rgb.astype(np.float32) / 255.0
    mx, mn = x.max(axis=2), x.min(axis=2)
    delta = mx - mn
    hue = np.zeros(mx.shape, dtype=np.float32)
    # Compute hue separately for the channel that is brightest at each pixel.
    nonzero = delta > 1e-8
    red = nonzero & (mx == x[..., 0])
    green = nonzero & (mx == x[..., 1])
    blue = nonzero & (mx == x[..., 2])
    hue[red] = ((x[..., 1][red] - x[..., 2][red]) / delta[red]) % 6
    hue[green] = (x[..., 2][green] - x[..., 0][green]) / delta[green] + 2
    hue[blue] = (x[..., 0][blue] - x[..., 1][blue]) / delta[blue] + 4
    hue = hue / 6.0
    saturation = np.divide(delta, mx, out=np.zeros_like(delta), where=mx > 1e-8)
    return hue, saturation, mx

# Build a mask from a loose green hue range plus an Excess Green score.
def make_green_mask(rgb):
    # A hue between about yellow-green and blue-green is a candidate plant color.
    hue, saturation, value = rgb_to_hsv(rgb)
    hue_green = (hue >= 0.16) & (hue <= 0.48) & (saturation >= 0.12) & (value >= 0.08)
    # Excess Green favors pixels where green is stronger than red and blue combined.
    red, green, blue = [rgb[..., i].astype(np.float32) for i in range(3)]
    excess_green = 2 * green - red - blue
    exg_green = (excess_green >= GREEN_EXG_MIN) & (green >= GREEN_G_DOMINANCE * (red + 1))
    return hue_green | exg_green

# Save each mask at preview size and record its pixel share and bounding box.
feature_rows = []
for _, row in images.iterrows():
    rgb = np.asarray(Image.open(SOURCE_ROOT / row['rgb_path']).convert('RGB'))
    mask = make_green_mask(rgb)
    height, width = mask.shape
    scale = PREVIEW_WIDTH / width
    preview_size = (PREVIEW_WIDTH, max(1, int(round(height * scale))))
    mask_image = Image.fromarray((mask * 255).astype(np.uint8)).resize(preview_size, Image.Resampling.NEAREST)
    mask_path = OUT_ROOT / 'vision/masks' / f"{row['image_id']}.png"
    mask_image.save(mask_path)
    y, x = np.nonzero(mask)
    bbox_width = int(x.max() - x.min() + 1) if len(x) else 0
    bbox_height = int(y.max() - y.min() + 1) if len(y) else 0
    feature_rows.append({
        'image_id': row['image_id'], 'mask_pixel_fraction': float(mask.mean()),
        'mask_bbox_width_px': bbox_width, 'mask_bbox_height_px': bbox_height,
        'mask_bbox_aspect_ratio': float(bbox_width / bbox_height) if bbox_height else np.nan,
        'mask_pixels': int(mask.sum()), 'mask_preview_path': f'vision/masks/{row["image_id"]}.png'
    })

features = images[['image_id','experiment_id','compartment_id','camera_id','ts_utc','local_time','split','chronological_split','rgb_path','depth_path','depth_dtype','depth_unit','valid_depth_fraction','calibration_id','calibration_status']].merge(pd.DataFrame(feature_rows), on='image_id', validate='one_to_one')
print('Created image-space masks for', len(features), 'pairs.')

## Cell 6 — Summarize native depth codes inside the mask

For the pixels selected by the color mask, this cell reports raw depth-code percentiles and the share equal to zero or 65535. A **percentile** is a point in the sorted values (for example, the 50th percentile is the middle value). These values stay in the camera file's native units, whose physical meaning is still unverified. No point-cloud projection or metric geometry is attempted.

In [ ]:
# Read each original depth image and summarize codes under its color mask.
depth_rows = []
for _, row in images.iterrows():
    rgb = np.asarray(Image.open(SOURCE_ROOT / row['rgb_path']).convert('RGB'))
    depth = np.asarray(Image.open(SOURCE_ROOT / row['depth_path']))
    mask = make_green_mask(rgb)
    selected = depth[mask]
    # Include every selected code in percentiles; separately report special code shares.
    depth_rows.append({
        'image_id': row['image_id'],
        'mask_depth_code_count': int(selected.size),
        'mask_depth_zero_fraction': float(np.mean(selected == 0)) if selected.size else np.nan,
        'mask_depth_65535_fraction': float(np.mean(selected == 65535)) if selected.size and depth.dtype == np.uint16 else np.nan,
        'mask_depth_code_p05': float(np.percentile(selected, 5)) if selected.size else np.nan,
        'mask_depth_code_p50': float(np.percentile(selected, 50)) if selected.size else np.nan,
        'mask_depth_code_p95': float(np.percentile(selected, 95)) if selected.size else np.nan,
    })

# Join depth summaries back to the image-space table.
features = features.merge(pd.DataFrame(depth_rows), on='image_id', validate='one_to_one')
features = features.merge(qc[['image_id','rgb_mean','rgb_max','qc_flags']], on='image_id', validate='one_to_one')
features.to_csv(OUT_ROOT / 'vision/spatial_features.csv', index=False)
try:
    # Parquet keeps typed columns compact; CSV remains the portable fallback.
    features.to_parquet(OUT_ROOT / 'vision/spatial_features.parquet', index=False)
except Exception as exc:
    warnings.warn(f'Parquet could not be written; CSV is included. Details: {exc}')
print(features[['image_id','mask_pixel_fraction','mask_depth_code_p50','qc_flags']].head().to_string(index=False))

## Cell 7 — Create a review montage and summary reports

The montage puts the color image, mask, and native-depth preview side by side for each pair. Depth contrast is stretched only for display; the saved depth source is untouched. The summary reports mask-size range and QC flags so you can spot empty masks or poor images before trusting the heuristic.

In [ ]:
# Create one row per pair: color preview, green mask, and display-scaled depth.
count = len(images)
fig, axes = plt.subplots(count, 3, figsize=(12, max(3, count * 1.25)))
for i, (_, row) in enumerate(images.iterrows()):
    rgb = np.asarray(Image.open(SOURCE_ROOT / row['rgb_path']).convert('RGB'))
    depth = np.asarray(Image.open(SOURCE_ROOT / row['depth_path']))
    mask_path = OUT_ROOT / 'vision/masks' / f"{row['image_id']}.png"
    mask_preview = np.asarray(Image.open(mask_path))
    # Percentile scaling makes the depth preview visible; it is display-only.
    lo, hi = np.percentile(depth, [2, 98])
    depth_preview = np.clip((depth.astype(np.float32) - lo) / max(hi - lo, 1), 0, 1)
    axes[i, 0].imshow(Image.fromarray(rgb).resize((PREVIEW_WIDTH, int(rgb.shape[0] * PREVIEW_WIDTH / rgb.shape[1]))))
    axes[i, 1].imshow(mask_preview, cmap='gray', vmin=0, vmax=255)
    axes[i, 2].imshow(depth_preview, cmap='viridis', vmin=0, vmax=1)
    axes[i, 0].set_ylabel(f"{row['compartment_id']} | {row['local_time']}", fontsize=7)
    for ax in axes[i]:
        ax.set_xticks([]); ax.set_yticks([])
for ax, title in zip(axes[0], ['RGB', 'Green mask', 'Depth preview (display only)']):
    ax.set_title(title)
fig.tight_layout()
fig.savefig(OUT_ROOT / 'reports/spatial_review_montage.png', dpi=100)
plt.close(fig)

# Write short summary files for quick review and later processing.
summary = {
    'producer': 'EDEN-03', 'code_version': '1.0', 'input_status': source_manifest['status'],
    'input_pairs': int(len(images)), 'numeric_calibration_verified': bool(numeric_calibration_verified),
    'depth_unit': depth_unit, 'metric_geometry_created': False,
    'qc_flagged_images': int((qc.qc_flags != 'none').sum()),
    'empty_masks': int((features.mask_pixel_fraction == 0).sum()),
    'mask_pixel_fraction_min': float(features.mask_pixel_fraction.min()),
    'mask_pixel_fraction_median': float(features.mask_pixel_fraction.median()),
    'mask_pixel_fraction_max': float(features.mask_pixel_fraction.max()),
    'pending': ['Human review of RGB/mask/depth montage', 'Green-mask threshold review',
                'Depth units and special-code meaning', 'Numeric camera intrinsics and reference plane']
}
(OUT_ROOT / 'reports/spatial_processing_report.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
qc.to_csv(OUT_ROOT / 'reports/image_quality.csv', index=False)
print(json.dumps(summary, indent=2))

## Cell 8 — Apply the acceptance gate and build the named ZIP

The gate checks that all 36 pairs produced features and masks and that every output is inventoried with a SHA-256 fingerprint. QC flags do not get hidden: they produce a partial status. Even with no flags, this is only a relative-feature pilot; human review and calibration checks are still required. The ZIP is saved outside its own source folder so it cannot include itself.

In [ ]:
# Require one output feature row and one saved mask for every input pair.
assert len(features) == len(images) == 36
assert features['image_id'].is_unique
for mask_path in (OUT_ROOT / 'vision/masks').glob('*.png'):
    assert mask_path.stat().st_size > 0, f'Empty mask file: {mask_path}'
assert len(list((OUT_ROOT / 'vision/masks').glob('*.png'))) == len(images)

# Record limitations and set a status that does not claim scientific validation.
has_qc_flags = bool((qc.qc_flags != 'none').any())
final_status = 'PARTIAL_PASS_QC_FLAGS_PRESENT' if has_qc_flags else 'PASS_RELATIVE_FEATURE_PILOT'
output_files = sorted(p for p in OUT_ROOT.rglob('*') if p.is_file() and p.name != 'manifest.json')
manifest = {
    'schema_version': '1.0', 'producer': 'EDEN-03', 'code_version': '1.0',
    'status': final_status, 'generated_utc': pd.Timestamp.now(tz='UTC').isoformat(),
    'input_manifest_sha256': hashlib.sha256((SOURCE_ROOT / 'manifest.json').read_bytes()).hexdigest(),
    'input_status': source_manifest['status'], 'input_pairs': int(len(images)),
    'numeric_calibration_verified': bool(numeric_calibration_verified), 'depth_unit': depth_unit,
    'metric_geometry_created': False, 'notes': 'Image-space heuristic features and native depth codes only; human review pending.',
    'files': []
}
for path in output_files:
    manifest['files'].append({
        'path': path.relative_to(OUT_ROOT).as_posix(), 'size': path.stat().st_size,
        'sha256': hashlib.sha256(path.read_bytes()).hexdigest()
    })
(OUT_ROOT / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

# Package the outputs using the agreed, versioned filename.
result_zip = WORK_ROOT / RESULT_NAME
if result_zip.exists():
    result_zip.unlink()
with zipfile.ZipFile(result_zip, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=3) as archive:
    for path in sorted(p for p in OUT_ROOT.rglob('*') if p.is_file()):
        archive.write(path, arcname=f'eden/{path.relative_to(OUT_ROOT).as_posix()}')
print('Status:', final_status)
print('Download this file:', result_zip)

## Run, review, and hand off

1. In Kaggle, create a new notebook, choose **CPU**, turn **Internet Off**, and attach the EDEN-02 result ZIP as an input dataset.
2. Paste/run all cells in order. Do not change input pair files or split labels.
3. Download `EDEN-03-Spatial-Processing-Results-v1.0.zip` from `/kaggle/working/` and upload it here.
4. Review `reports/spatial_review_montage.png` and `reports/image_quality.csv`. In particular, check the two dark frames from EDEN-02 and any empty or oversized masks.

**Troubleshooting:** If the result is not found, attach the specifically named EDEN-02 ZIP. If a hash fails, re-download the upstream result rather than bypassing verification. If the montage shows too much background or missed leaves, report which rows; threshold changes should be versioned. Do not interpret depth codes as millimetres or claim physical plant size until the calibration evidence is verified.

Source: Maree et al. (2025), [4th Autonomous Greenhouse Challenge: Dwarf Tomato Timeseries and Images, v1](https://doi.org/10.4121/fa102772-32db-4b30-bace-12f2016722ce.v1), CC BY 4.0.